# AKLT Ground State With Imaginary-Time iTEBD

This notebook demonstrates a standard imaginary-time iTEBD calculation.
We evolve a random spin-1 state toward the AKLT ground state, then visualize how the overlap, energy density, and entanglement converge.

## Setup

In [ ]:
import Pkg
Pkg.activate("..")

using iTEBD, LinearAlgebra, Plots

default(linewidth=2, legend=:right, framestyle=:box, size=(900, 300))

## AKLT Hamiltonian And Exact State

In [ ]:
X = sqrt(2) / 2 * [0 1 0; 1 0 1; 0 1 0]
Y = sqrt(2) / 2 * 1im * [0 -1 0; 1 0 -1; 0 1 0]
Z = [1 0 0; 0 0 0; 0 0 -1]

SS = kron(X, X) + kron(Y, Y) + kron(Z, Z)
H = 0.5 * SS + SS^2 / 6 + I / 3
d_tau = 0.1
G = exp(-d_tau * H)

aklt_tensor = zeros(ComplexF64, 2, 3, 2)
aklt_tensor[1, 1, 2] = sqrt(2 / 3)
aklt_tensor[1, 2, 1] = -sqrt(1 / 3)
aklt_tensor[2, 2, 2] = sqrt(1 / 3)
aklt_tensor[2, 3, 1] = -sqrt(2 / 3)
aklt = iMPS([aklt_tensor, aklt_tensor])

bond_energy(psi) = 0.5 * (iTEBD.expect(psi, H, 1, 2) + iTEBD.expect(psi, H, 2, 1))

## Imaginary-Time Evolution

Each iteration applies the two-site imaginary-time gate to both bonds of the two-site unit cell.

In [ ]:
psi = rand_iMPS(ComplexF64, 2, 3, 1)
steps = 300

overlap_hist = zeros(steps)
energy_hist = zeros(steps)
entropy_hist = zeros(steps)

for step in 1:steps
    applygate!(psi, G, 1, 2; maxdim=8)
    applygate!(psi, G, 2, 1; maxdim=8)

    overlap_hist[step] = inner_product(aklt, psi)
    energy_hist[step] = bond_energy(psi)
    entropy_hist[step] = iTEBD.ent_S(psi, 1)
end

In [ ]:
(;
    final_overlap=overlap_hist[end],
    final_energy=energy_hist[end],
    exact_energy=bond_energy(aklt),
    final_entropy=entropy_hist[end],
)

In [ ]:
p1 = plot(1:steps, overlap_hist, xlabel="step", ylabel="overlap", label="with exact AKLT")
p2 = plot(1:steps, energy_hist, xlabel="step", ylabel="energy density", label="evolved")
hline!(p2, [bond_energy(aklt)], linestyle=:dash, label="exact")
p3 = plot(1:steps, entropy_hist, xlabel="step", ylabel="entanglement", label="evolved")

plot(p1, p2, p3; layout=(1, 3), size=(1200, 320))

## Adaptive Bond-Dimension Evolution

The same imaginary-time sweep can be expressed with the high-level `evolve!` wrapper. Setting `chi_policy=:adaptive` keeps the bond dimension non-decreasing while letting the Schmidt spectrum choose a natural value up to `maxdim`.

In [ ]:
psi_adaptive = rand_iMPS(ComplexF64, 2, 3, 1)
gates = [(G, 1, 2), (G, 2, 1)]

evolve!(psi_adaptive, gates, steps; chi_policy=:adaptive, maxdim=8)

(;
    adaptive_overlap=inner_product(aklt, psi_adaptive),
    adaptive_energy=bond_energy(psi_adaptive),
    adaptive_maxbond=maximum(length.(psi_adaptive.λ)),
    adaptive_lambdas=psi_adaptive.λ,
)
